In [ ]:
# !pip install requests beautifulsoup4 lxml jupyter

In [ ]:
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseAdvancedSearch&searchAnnCons"
SEARCH_PATH = "/pmmp/"  # TODO: à ajuster si le vrai chemin diffère

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
})

print("Session initialisée.")

Session initialisée.


### Fonctions utilitaires (extraction de formulaire + parsing de liste)

## Etape 1 : GET du formulaire( etablir les cookies du session)

In [34]:
resp_form = session.get(BASE_URL + SEARCH_PATH, timeout=20)
print(resp_form.status_code)
print(session.cookies.get_dict())

200
{'PHPSESSID': 'f02d04538362987c1415fcc90b0ce56e', 'SERVERID': 'server15', 'TS01fd0ca2': '01a853b005e1cce44aac4e875eb8a54987208bdb309f00b8bd666351d665c56f55c90689368979d492041789d6d1e7a91f3ae8bc643e7271846607dedb24b010e8a602b9d72f290c7edea61f2c219e17ac99333e52'}


###  Parsing + sauvegarde du HTML du formulaire

In [41]:
soup_form = BeautifulSoup(resp_form.text, "lxml")
print("Formulaire chargé, taille HTML:", len(resp_form.text))

with open("debug_formulaire.html", "w", encoding="utf-8") as f:
    f.write(resp_form.text)

print("Fichier sauvegardé : debug_formulaire.html")

Formulaire chargé, taille HTML: 86283
Fichier sauvegardé : debug_formulaire.html


###  Inspection ET extraction complète des champs du formulaire

**C'est la cellule clé, fusionnée : elle affiche les champs pour vérification visuelle, et construit directement form_data, le dict complet à renvoyer au serveur (PRADO exige tous les champs, y compris PRADO_PAGESTATE, sinon le postback échoue).**

In [47]:
def extract_form_fields(soup):
    data = {}
    for tag in soup.find_all(["input", "select", "textarea"]):
        name = tag.get("name")
        if not name:
            continue
        ttype = (tag.get("type") or "text").lower()
        if ttype in ("submit", "image", "reset", "button"):
            continue  # on ne les inclut que si on veut cliquer dessus explicitement
        if ttype in ("checkbox", "radio"):
            if tag.has_attr("checked"):
                data[name] = tag.get("value", "on")
            continue
        if tag.name == "select":
            selected = tag.find("option", selected=True)
            data[name] = selected.get("value") if selected else ""
            continue
        data[name] = tag.get("value", "")
    return data

form_data = extract_form_fields(soup_form)

print(len(form_data), "champs extraits")
print("PRADO_PAGESTATE présent :", "PRADO_PAGESTATE" in form_data)

# Inspection visuelle (optionnelle mais utile pour repérer un champ manquant/inattendu)
for name, value in form_data.items():
    apercu = (value[:40] + "...") if value and len(value) > 40 else value
    print(name, "=", apercu)

25 champs extraits
PRADO_PAGESTATE présent : True
PRADO_PAGESTATE = eJztfdty20iSaH8KQjsxckdYNgHwKo9ng6Jom926...
PRADO_POSTBACK_TARGET = 
PRADO_POSTBACK_PARAMETER = 
ctl0$menuGaucheEntreprise$quickSearch = Recherche rapide
ctl0$CONTENU_PAGE$AdvancedSearch$type_rechercheEntite = floue
ctl0$CONTENU_PAGE$AdvancedSearch$classification = 
ctl0$CONTENU_PAGE$AdvancedSearch$organismesNames = 
ctl0$CONTENU_PAGE$AdvancedSearch$entityPurchaseNames = 
ctl0$CONTENU_PAGE$AdvancedSearch$choixInclusionDescendancesServices = ctl0$CONTENU_PAGE$AdvancedSearch$inclure...
ctl0$CONTENU_PAGE$AdvancedSearch$orgName = 
ctl0$CONTENU_PAGE$AdvancedSearch$reference = 
ctl0$CONTENU_PAGE$AdvancedSearch$annonceType = 
ctl0$CONTENU_PAGE$AdvancedSearch$procedureType = 
ctl0$CONTENU_PAGE$AdvancedSearch$categorie = 
ctl0$CONTENU_PAGE$AdvancedSearch$idsSelectedGeoN2 = 
ctl0$CONTENU_PAGE$AdvancedSearch$numSelectedGeoN2 = 
ctl0$CONTENU_PAGE$AdvancedSearch$domaineActivite$idsDomaines = 
ctl0$CONTENU_PAGE$AdvancedSearch$qualif

##  Étape 2 : POST de la recherche (postback du bouton "Lancer la recherche")

In [52]:
form_data["ctl0$CONTENU_PAGE$AdvancedSearch$lancerRecherche"] = "Lancer la recherche"

resp_list = session.post(BASE_URL + SEARCH_PATH, data=form_data, timeout=20)
print(resp_list.status_code)

with open("debug_liste.html", "w", encoding="utf-8") as f:
    f.write(resp_list.text)

soup_list = BeautifulSoup(resp_list.text, "lxml")
print("Taille HTML retourné :", len(resp_list.text))

200
Taille HTML retourné : 258169


### Inspection de la structure de la page de résultats

In [56]:
tables = soup_list.find_all("table")
print(f"{len(tables)} table(s) trouvée(s)")

for i, t in enumerate(tables):
    print(f"--- table {i} --- classes: {t.get('class')}, {len(t.find_all('tr'))} lignes")

1 table(s) trouvée(s)
--- table 0 --- classes: ['table-results'], 12 lignes


### Extraction d'une ligne test (ajuste TABLE_INDEX selon la cellule 7)


In [59]:
# TODO: remplace 0 par l'indice de la vraie table de résultats trouvée en cellule 7
TABLE_INDEX = 0
rows = tables[TABLE_INDEX].find_all("tr")
print(f"{len(rows)} lignes trouvées")

if len(rows) > 1:
    first_row = rows[1]  # rows[0] est souvent l'en-tête
    cells = first_row.find_all("td")
    for i, c in enumerate(cells):
        print(i, "|", c.get_text(strip=True))

12 lignes trouvées


### Exploration du mécanisme d'ouverture d'une fiche details(le point le plus incertain)

##### Objectif : repérer, dans une ligne de résultat, le lien ou le formulaire qui déclenche l'ouverture de la fiche détail (pas d'URL directe attendue).

In [61]:
if len(rows) > 1:
    first_row = rows[1]
    links = first_row.find_all("a")
    for l in links:
        print("href:", l.get("href"), "| onclick:", l.get("onclick"))

    forms_in_row = first_row.find_all("form")
    for f in forms_in_row:
        print("form action:", f.get("action"))

href: javascript:void(0); | onclick: None


In [63]:
# rows[0] est l'en-tête (confirmé). On regarde une vraie ligne de résultat.
data_row = rows[1]
print(data_row.prettify())

<tr>
 <th abbr="Tous" class="check-col" id="checkAll" style="display:none">
  <input id="ctl0_CONTENU_PAGE_resultSearch_tableauResultSearch_ctl0_selectAll" name="ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl0$selectAll" onclick="CheckUnCheckAll(this, 'resultSearch_tableauResultSearch', 'annonceSelection');" title="Tous/Aucun" type="checkbox"/>
 </th>
 <th abbr="Ref" class="col-90" id="cons_ref">
  Procédure
  <br/>
  Catégorie
  <input alt="Trier" id="ctl0_CONTENU_PAGE_resultSearch_tableauResultSearch_ctl0_ctl3" name="ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl0$ctl3" src="themes/images/arrow-tri-off.gif" title="Trier Par 'Catégorie'" type="image"/>
  <br/>
  Publié le
  <input alt="Trier" id="ctl0_CONTENU_PAGE_resultSearch_tableauResultSearch_ctl0_ctl5" name="ctl0$CONTENU_PAGE$resultSearch$tableauResultSearch$ctl0$ctl5" src="themes/images/arrow-tri-off.gif" title="Trier Par 'Publié le'" type="image"/>
 </th>
 <th abbr="Intitule" class="col-450" id="cons_intitule">


### recherche global du pattern

In [64]:
import re

matches = re.findall(r"__doPostBack\('([^']+)'\s*,\s*'([^']*)'\)", resp_list.text)
print(f"{len(matches)} occurrences de __doPostBack trouvées")
for target, arg in matches[:20]:
    print(target, "|", arg)

0 occurrences de __doPostBack trouvées


In [65]:
print("rows[0] est un header ?", len(rows[0].find_all("th")) > 0)
print("rows[1] est un header ?", len(rows[1].find_all("th")) > 0)
print("rows[1] contient des td ?", len(rows[1].find_all("td")))

rows[0] est un header ? True
rows[1] est un header ? True
rows[1] contient des td ? 0


In [66]:
data_row = rows[1]

print("=== Liens <a> ===")
for a in data_row.find_all("a"):
    print("href:", a.get("href"), "| onclick:", a.get("onclick"), "| texte:", a.get_text(strip=True))

print("\n=== Éléments avec onclick (tout type) ===")
for el in data_row.find_all(attrs={"onclick": True}):
    print(el.name, "| onclick:", el.get("onclick"))

print("\n=== Colonne Actions (dernière <td>) ===")
tds = data_row.find_all("td")
if tds:
    print(tds[-1].prettify())

=== Liens <a> ===
href: javascript:void(0); | onclick: None | texte: Clauses soc./env.

=== Éléments avec onclick (tout type) ===
input | onclick: CheckUnCheckAll(this, 'resultSearch_tableauResultSearch', 'annonceSelection');

=== Colonne Actions (dernière <td>) ===


In [67]:
import re

patterns = [
    r"Prado\.Callback\([^)]*\)",
    r"WebForm_DoCallback\([^)]*\)",
    r"onclick=\"([^\"]*detail[^\"]*)\"",
    r"onclick=\"([^\"]*Detail[^\"]*)\"",
]

for pat in patterns:
    found = re.findall(pat, resp_list.text)
    print(f"Pattern '{pat}' : {len(found)} occurrence(s)")
    for f in found[:5]:
        print("  ", f)

Pattern 'Prado\.Callback\([^)]*\)' : 0 occurrence(s)
Pattern 'WebForm_DoCallback\([^)]*\)' : 0 occurrence(s)
Pattern 'onclick=\"([^\"]*detail[^\"]*)\"' : 0 occurrence(s)
Pattern 'onclick=\"([^\"]*Detail[^\"]*)\"' : 0 occurrence(s)


In [68]:
for i, r in enumerate(rows):
    th_count = len(r.find_all("th"))
    td_count = len(r.find_all("td"))
    print(i, "| th:", th_count, "| td:", td_count)

0 | th: 1 | td: 0
1 | th: 6 | td: 0
2 | th: 0 | td: 7
3 | th: 0 | td: 7
4 | th: 0 | td: 7
5 | th: 0 | td: 7
6 | th: 0 | td: 7
7 | th: 0 | td: 7
8 | th: 0 | td: 7
9 | th: 0 | td: 7
10 | th: 0 | td: 7
11 | th: 0 | td: 7


In [69]:
# TODO: remplace N par le premier indice avec td_count > 0 trouvé en cellule 15
N = 2
data_row = rows[N]

print("=== Liens <a> ===")
for a in data_row.find_all("a"):
    print("href:", a.get("href"), "| onclick:", a.get("onclick"), "| texte:", a.get_text(strip=True)[:60])

print("\n=== Éléments avec onclick ===")
for el in data_row.find_all(attrs={"onclick": True}):
    print(el.name, "| onclick:", el.get("onclick"))

print("\n=== Toutes les td ===")
for i, td in enumerate(data_row.find_all("td")):
    print(i, "|", td.get_text(strip=True)[:60])

=== Liens <a> ===
href: javascript:void(0); | onclick: None | texte: ...
href: None | onclick: None | texte: 
href: javascript:popUp('index.php?page=commun.PopUpDetailLots&orgAccronyme=g3h&refConsultation=1023160&lang=','yes') | onclick: None | texte: 
href: javascript:void(0); | onclick: None | texte: ...
href: ?page=entreprise.EntrepriseDetailConsultation&refConsultation=1023160&orgAcronyme=g3h | onclick: None | texte: 
href: index.php?page=commun.DiagnosticPoste&callFrom=entreprise | onclick: None | texte: 
href: index.php?page=entreprise.EntrepriseHome&goto=%3Fpage%3Dentreprise.EntrepriseAccueilAuthentifie | onclick: None | texte: 
href: https://www.marchespublics.gov.ma/?page=entreprise.EntrepriseDetailsConsultation&refConsultation=1023160&orgAcronyme=g3h&code=&retraits | onclick: None | texte: 0
href: https://www.marchespublics.gov.ma/?page=entreprise.EntrepriseDetailsConsultation&refConsultation=1023160&orgAcronyme=g3h&code=&questions | onclick: None | texte: 0
href: https://www

In [70]:
import re

def extract_row_data(row):
    tds = row.find_all("td")
    if len(tds) < 7:
        return None

    detail_link = row.find("a", href=re.compile(r"page=entreprise\.EntrepriseDetailConsultation"))
    ref_consultation = org_acronyme = None
    if detail_link:
        m = re.search(r"refConsultation=(\w+)", detail_link["href"])
        o = re.search(r"orgAcronyme=(\w+)", detail_link["href"])
        ref_consultation = m.group(1) if m else None
        org_acronyme = o.group(1) if o else None

    return {
        "procedure_categorie": tds[1].get_text(" ", strip=True),
        "reference_objet": tds[2].get_text(" ", strip=True),
        "organisme": tds[3].get_text(" ", strip=True),
        "date_limite": tds[4].get_text(" ", strip=True),
        "ref_consultation": ref_consultation,
        "org_acronyme": org_acronyme,
    }

# Applique sur toutes les vraies lignes de données (indices 2 à 11 d'après ta cellule 15)
data_rows = rows[2:]
extracted = [extract_row_data(r) for r in data_rows]
extracted = [e for e in extracted if e]

print(len(extracted), "lignes extraites")
for e in extracted[:3]:
    print(e)

10 lignes extraites
{'procedure_categorie': "AOO ... Appel d'offres ouvert Travaux 14/07/2026", 'reference_objet': "25/AOO/AASLM/2026 - ... Objet\r\n                        : REALISATION DES TRAVAUX PREPARATOIRE DES PROJETS D'AMENAGEMENT DE L'AGENCE. ... REALISATION DES TRAVAUX PREPARATOIRE DES PROJETS D'AMENAGEMENT DE L'AGENCE. Acheteur public\r\n                        : AGENCE POUR L'AMENAGEMENT DU SITE DE LA LAGUNE DE MARCHICA", 'organisme': '- NADOR ... NADOR ...', 'date_limite': '13/07/2028 12:12 ...', 'ref_consultation': '1023160', 'org_acronyme': 'g3h'}
{'procedure_categorie': "AOO ... Appel d'offres ouvert Services 14/07/2026", 'reference_objet': '34/DPJ/2026AS - ... Objet\r\n                        : Assistance technique et suivi des travaux de  dédoublement du collecteur principal El Quods à la ville d’El Jadida ... Assistance technique et suivi des travaux de  dédoublement du collecteur principal El Quods à la ville d’El Jadida Acheteur public\r\n                        : S

In [74]:
ROOT_URL = "https://www.marchespublics.gov.ma"  # domaine seul, jamais concaténé à SEARCH_PATH

first = extracted[0]
detail_url = (
    f"{ROOT_URL}/index.php?page=entreprise.EntrepriseDetailConsultation"
    f"&refConsultation={first['ref_consultation']}&orgAcronyme={first['org_acronyme']}"
)
print("URL appelée :", detail_url)

resp_detail = session.get(detail_url, timeout=20, headers={"Referer": ROOT_URL + "/" + SEARCH_PATH.lstrip("/")})
print(resp_detail.status_code, len(resp_detail.text))

with open("debug_fiche.html", "w", encoding="utf-8") as f:
    f.write(resp_detail.text)

soup_detail = BeautifulSoup(resp_detail.text, "lxml")
print("Titre page:", soup_detail.title.get_text(strip=True) if soup_detail.title else None)

URL appelée : https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDetailConsultation&refConsultation=1023160&orgAcronyme=g3h
200 118162
Titre page: Marchés publics électroniques


In [76]:
retraits_url = (
    f"{ROOT_URL}/index.php?page=entreprise.EntrepriseDetailsConsultation"
    f"&refConsultation={first['ref_consultation']}&orgAcronyme={first['org_acronyme']}&retraits"
)
print("URL onglet retraits:", retraits_url)

resp_retraits = session.get(retraits_url, timeout=20, headers={"Referer": detail_url})
print(resp_retraits.status_code, len(resp_retraits.text))

soup_retraits = BeautifulSoup(resp_retraits.text, "lxml")
print("Titre page:", soup_retraits.title.get_text(strip=True) if soup_retraits.title else None)

doc_candidates = [a for a in soup_retraits.find_all("a", href=True)
                   if any(k in a["href"].lower() for k in [".pdf", ".zip", ".doc", "telecharg", "download"])]
print(f"{len(doc_candidates)} lien(s) document(s) trouvé(s)")
for a in doc_candidates[:20]:
    print(a.get_text(strip=True)[:50], "->", a["href"])

URL onglet retraits: https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDetailsConsultation&refConsultation=1023160&orgAcronyme=g3h&retraits
200 118137
Titre page: Marchés publics électroniques
2 lien(s) document(s) trouvé(s)
Fichier joint - Avis complémentaire en ligne -> index.php?page=entreprise.EntrepriseDownloadAvisJAL&refConsultation=1023160&orgAcronyme=g3h&idAvis=524330
 -> index.php?page=entreprise.EntrepriseDownloadAvisJAL&refConsultation=1023160&orgAcronyme=g3h&idAvis=


In [82]:
print(f"{len(soup_retraits.find_all('a', href=True))} liens au total sur la page 'retraits'\n")

for a in soup_retraits.find_all("a", href=True):
    texte = a.get_text(strip=True)
    if texte or "page=" in a["href"] or "panier" in a["href"].lower():
        print(texte[:50], "->", a["href"])

91 liens au total sur la page 'retraits'

Aller au menu -> #menu
Aller au contenu -> #main-part
Vous n'êtes pas authentifié -> javascript:;//ctl0_bandeauEntreprise_ctl1
PORTAIL -> pmmp
S'identifier -> javascript:;//ctl0_menuGaucheEntreprise_ctl0
Mon panier -> javascript:void(0);
Consultations -> javascript:void(0);
Toutes les consultations -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&retraits
sans retrait -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&sansRetraits
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&questions
sans question posée -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&sansQuestions
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&depots
sans dépôt -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&sansDepots
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&echanges
sans message échangé -> ?

In [83]:
print(f"{len(soup_retraits.find_all('a', href=True))} liens au total sur la page 'retraits'\n")

for a in soup_retraits.find_all("a", href=True):
    texte = a.get_text(strip=True)
    if texte or "page=" in a["href"] or "panier" in a["href"].lower():
        print(texte[:50], "->", a["href"])

91 liens au total sur la page 'retraits'

Aller au menu -> #menu
Aller au contenu -> #main-part
Vous n'êtes pas authentifié -> javascript:;//ctl0_bandeauEntreprise_ctl1
PORTAIL -> pmmp
S'identifier -> javascript:;//ctl0_menuGaucheEntreprise_ctl0
Mon panier -> javascript:void(0);
Consultations -> javascript:void(0);
Toutes les consultations -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&retraits
sans retrait -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&sansRetraits
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&questions
sans question posée -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&sansQuestions
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&depots
sans dépôt -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&sansDepots
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&echanges
sans message échangé -> ?

In [84]:
import re

# Liens <a> contenant "panier"
panier_links = [a for a in soup_retraits.find_all("a", href=True) if "panier" in a["href"].lower()]
print("=== Liens contenant 'panier' ===")
for a in panier_links:
    print(a.get_text(strip=True)[:50], "->", a["href"])

# Recherche des appels JS complets popUp(...) ou similaires liés au panier
js_calls = re.findall(r"(?:popUp|window\.open)\(['\"]([^'\"]*panier[^'\"]*)['\"]", resp_retraits.text, re.IGNORECASE)
print("\n=== Appels JS liés au panier ===")
for c in set(js_calls):
    print(c)

=== Liens contenant 'panier' ===
Toutes les consultations -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&retraits
sans retrait -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&sansRetraits
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&questions
sans question posée -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&sansQuestions
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&depots
sans dépôt -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&sansDepots
Avec -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&echanges
sans message échangé -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&sansEchanges
Toutes les consultations -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise
Consultations clôturées -> ?page=entreprise.EntrepriseAdvancedSearch&panierEntreprise&panierEntreprise&dateFinEnd
Recherche av

In [86]:
import re

zip_mentions = re.findall(r'[^"\'\s]*\.zip[^"\'\s]*', resp_retraits.text, re.IGNORECASE)
print(f"{len(zip_mentions)} mention(s) de .zip trouvée(s)")
for z in set(zip_mentions):
    print(z)

0 mention(s) de .zip trouvée(s)


In [87]:
keywords = ["retir", "dossier", "telecharg", "download", "dce"]

print("=== Liens <a> ===")
for a in soup_retraits.find_all("a", href=True):
    texte = a.get_text(strip=True).lower()
    href = a["href"].lower()
    if any(k in texte or k in href for k in keywords):
        print(a.get_text(strip=True)[:60], "->", a["href"])

print("\n=== Boutons <input type=submit/button> ===")
for btn in soup_retraits.find_all("input", type=["submit", "button"]):
    val = (btn.get("value") or "").lower()
    name = (btn.get("name") or "").lower()
    if any(k in val or k in name for k in keywords):
        print("name:", btn.get("name"), "| value:", btn.get("value"))

print("\n=== Formulaires de la page ===")
for f in soup_retraits.find_all("form"):
    print("action:", f.get("action"), "| id:", f.get("id"))

=== Liens <a> ===
Fichier joint - Avis complémentaire en ligne -> index.php?page=entreprise.EntrepriseDownloadAvisJAL&refConsultation=1023160&orgAcronyme=g3h&idAvis=524330
 -> index.php?page=entreprise.EntrepriseDownloadAvisJAL&refConsultation=1023160&orgAcronyme=g3h&idAvis=

=== Boutons <input type=submit/button> ===
name: ctl0$CONTENU_PAGE$DefaultButtonTopTelechargement | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonBottomTelechargement | value: 
name: ctl0$CONTENU_PAGE$refreshRepeaterTelechargement | value: 

=== Formulaires de la page ===
action: /index.php?page=entreprise.EntrepriseDetailsConsultation&refConsultation=1023160&orgAcronyme=g3h&retraits | id: ctl0_ctl4


In [88]:
# TODO: ajuste l'index si plusieurs formulaires sont trouvés
forms = soup_retraits.find_all("form")
if forms:
    target_form = forms[0]  # à ajuster selon la cellule 24
    for tag in target_form.find_all(["input", "select", "textarea"]):
        print(tag.get("name"), "|", tag.get("type"), "|", tag.get("value"))

PRADO_PAGESTATE | hidden | eJztfdlyHDeyqF/PX1TwzAzlCFPqWnqjxj7RIqnFI5I9bEpz474oit3oJqzqqnYtlDQT+pfzePR2vuHqxy4SqCWxVHWRokRqXA5LdqOQQCKRyA0JwHH9UW85XPQulk7/wvPH/b4/WMydntMfOezLwnc8shwt/X17/190v/fY33f2/5Xse/s78zTo7bDfQ+V3b/9fH4sCWy1woKBsygbQ4f7Oa5rQi4DsPL7Ytx9/LCu716nsQWXHrpDpS9Ac6/H+zmwe0036Kg52HrPf/f2dyzTdJPuPHr179+7h2o/nlyTZZBcBnScPV9EVK3r0KCZJEmXxnCSP5n5AwkVMSfzooPzfZ/84l389DKO5z1p6+FsC3chI9xDSg+042j33VpA8IySQEc1LWiM7bIGsczvIThZrGtIkjf2URqGMtvatxQBsZ3/nwg8XxM8mKxKmfCR2PoKNH5LgZbSK2rGb7RYgNHwbtADDNEQrxBnk7Zz4V3SV8dEcZwENaLjKCNRzK2ZGC2m0vxP44cpfkUSp1NtBCCwDf/VivZKQ43UZ/Is1g86nzGFrIL0kazYVFIqTRwC5t4wfbsIVn1N7f+cgWq8Z+U78NeEwrPlYMOf+ziRNY3qRpYDO6b7DSs7LooMoCMgcRrbDcGSdu2zQP5i+//Bm7idkRsKEpvQqpxtH9ofzY3/zw5vFTsF0jP5+kOZ4PD2D/4Ex0DQgqPAjBo6L9mwPythwNlHI+OCHN6QgLPwjUfuWCOlvIaR/Twg5MRFyckNCIhHBYA/91P8b+cCHxTo4f8lWbz6MMbQCvyu8AFb8sfmfj1Kt+Q4rdB5LZW2Q4gAvUrI+iDJY/9BIgWfjUk/IIZlHYcioS2JojuNtwypknTNCx1d0To5JSsVnpIGc8jfTSzsx2RCftfESgSUV0Yo2w7fHUQhDSEklqqo2XTQipPigZIsAKiQdmw1SNgoUW7CCSfYbk8eLy4yi9

In [89]:
import re

keywords = ["nom", "prenom", "email", "mail", "condition", "civil"]

# Cherche parmi TOUS les inputs de la page actuelle (soup_retraits ou la dernière page récupérée)
current_soup = soup_retraits  # TODO: remplace par la variable de la page où tu as vu ce formulaire

for tag in current_soup.find_all(["input", "select", "textarea"]):
    name = (tag.get("name") or "").lower()
    if any(k in name for k in keywords):
        print(tag.get("name"), "|", tag.get("type"), "|", tag.get("value"))

In [90]:
for btn in current_soup.find_all("input", type="submit"):
    print("name:", btn.get("name"), "| value:", btn.get("value"))

name: ctl0$CONTENU_PAGE$DefaultButtonTopTelechargement | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonBottomTelechargement | value: 
name: ctl0$CONTENU_PAGE$refreshRepeaterTelechargement | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonTopQuestion | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonBottomQuestion | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonTopCaution | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonBottomCaution | value: 
name: ctl0$CONTENU_PAGE$refreshRepeaterCaution | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonTopMessagerie | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonBottomMessagerie | value: 
name: ctl0$CONTENU_PAGE$afficherMonPanier | value: 
name: ctl0$CONTENU_PAGE$ctl75$ctl1 | value: Valider
name: ctl0$CONTENU_PAGE$popinTitle | value: 
name: ctl0$CONTENU_PAGE$validerSelectionLots | value: Valider


In [91]:
form_data_lots = extract_form_fields(current_soup)
form_data_lots["ctl0$CONTENU_PAGE$validerSelectionLots"] = "Valider"

resp_step2 = session.post(retraits_url, data=form_data_lots, timeout=20)
print(resp_step2.status_code, len(resp_step2.text))

with open("debug_step2.html", "w", encoding="utf-8") as f:
    f.write(resp_step2.text)

soup_step2 = BeautifulSoup(resp_step2.text, "lxml")
print("Titre:", soup_step2.title.get_text(strip=True) if soup_step2.title else None)

200 118343
Titre: Marchés publics électroniques


In [92]:
for tag in soup_step2.find_all(["input", "select", "textarea"]):
    name = (tag.get("name") or "").lower()
    if any(k in name for k in keywords):
        print(tag.get("name"), "|", tag.get("type"), "|", tag.get("value"))

print("\n=== Boutons submit ===")
for btn in soup_step2.find_all("input", type="submit"):
    print("name:", btn.get("name"), "| value:", btn.get("value"))


=== Boutons submit ===
name: ctl0$CONTENU_PAGE$DefaultButtonTopTelechargement | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonBottomTelechargement | value: 
name: ctl0$CONTENU_PAGE$refreshRepeaterTelechargement | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonTopQuestion | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonBottomQuestion | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonTopCaution | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonBottomCaution | value: 
name: ctl0$CONTENU_PAGE$refreshRepeaterCaution | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonTopMessagerie | value: 
name: ctl0$CONTENU_PAGE$DefaultButtonBottomMessagerie | value: 
name: ctl0$CONTENU_PAGE$afficherMonPanier | value: 
name: ctl0$CONTENU_PAGE$ctl75$ctl1 | value: Valider
name: ctl0$CONTENU_PAGE$popinTitle | value: 
name: ctl0$CONTENU_PAGE$validerSelectionLots | value: Valider


In [93]:
for tag in current_soup.find_all(["input", "th", "td"]):
    name = (tag.get("name") or "").lower()
    if "lot" in name:
        print(tag.name, "|", tag.get("name"), "|", tag.get("type"), "|", tag.get("value"))

input | ctl0$CONTENU_PAGE$multiSelectLots$dataJson | hidden | None
input | ctl0$CONTENU_PAGE$multiSelectLots$selectedData | hidden | None
input | ctl0$CONTENU_PAGE$validerSelectionLots | submit | Valider


In [94]:
checkboxes = current_soup.find_all("input", type="checkbox")
print(f"{len(checkboxes)} checkbox(es) trouvée(s)")
for c in checkboxes:
    print(c.get("name"), "| value:", c.get("value"), "| checked:", c.has_attr("checked"))

0 checkbox(es) trouvée(s)


In [95]:
checkboxes = current_soup.find_all("input", type="checkbox")
print(f"{len(checkboxes)} checkbox(es) trouvée(s)")
for c in checkboxes:
    print(c.get("name"), "| value:", c.get("value"), "| checked:", c.has_attr("checked"))

0 checkbox(es) trouvée(s)


In [96]:
# Cherche le script qui définit ou initialise multiSelectLots
import re
scripts = current_soup.find_all("script")
for s in scripts:
    if s.string and "multiSelectLots" in s.string:
        print(s.string[:2000])
        print("---")


    J(document).ready(function () {
        let opts = JSON.stringify([]);
        VirtualSelect.init({
            ele: "#ctl0_CONTENU_PAGE_multiSelectLots-multiselect",
            options: JSON.parse(opts),
            multiple: true,
            zIndex: 999,
            dropboxWidth : '100%',
            showValueAsTags: true,
            showSelectedOptionsFirst: true,
            searchPlaceholderText: "Rechercher par lot",
            noSearchResultsText: "Aucun résultat",
            moreText: "de plus...",
            selectedValue: "".split(',')
        });

        J('#ctl0_CONTENU_PAGE_multiSelectLots-multiselect').change(function () {
            J('#' + "ctl0_CONTENU_PAGE_multiSelectLots_selectedData").val(this.value);
            J('#ctl0_CONTENU_PAGE_multiSelectLots-multiselect .vscomp-value-tag').each(function () {
                let tagElt = J(this);
                let tagContent = tagElt.find('.vscomp-value-tag-content').text();
                tagElt.attr('title'

### Sauver le HTML complet et chercher TOUTES les occurrences de "multiSelectLots" (pas juste la première)

In [98]:
with open("debug_step2_full.html", "w", encoding="utf-8") as f:
    f.write(resp_step2.text)

import re
positions = [m.start() for m in re.finditer("multiSelectLots", resp_step2.text)]
print(f"{len(positions)} occurrence(s) de 'multiSelectLots' dans la page")
for p in positions:
    print("---")
    print(resp_step2.text[max(0, p-100):p+200])

13 occurrence(s) de 'multiSelectLots' dans la page
---
 lots : </h3>
					 </div>

					 <div class="multiselect-wrapper">
    <div id="ctl0_CONTENU_PAGE_multiSelectLots-multiselect"
         placeholder="Sélectionner"

    ></div>
</div>

<input type="hidden" name="ctl0$CONTENU_PAGE$multiSelectLots$dataJson" id="ctl0_CONTENU_PAGE_multiSelectLots
---
  placeholder="Sélectionner"

    ></div>
</div>

<input type="hidden" name="ctl0$CONTENU_PAGE$multiSelectLots$dataJson" id="ctl0_CONTENU_PAGE_multiSelectLots_dataJson" />
<input type="hidden" name="ctl0$CONTENU_PAGE$multiSelectLots$selectedData" id="ctl0_CONTENU_PAGE_multiSelectLots_selectedD
---
iv>

<input type="hidden" name="ctl0$CONTENU_PAGE$multiSelectLots$dataJson" id="ctl0_CONTENU_PAGE_multiSelectLots_dataJson" />
<input type="hidden" name="ctl0$CONTENU_PAGE$multiSelectLots$selectedData" id="ctl0_CONTENU_PAGE_multiSelectLots_selectedData" />

<script>
    J(document).ready(funct
---
n" id="ctl0_CONTENU_PAGE_multiSelectLots_dataJson

In [99]:
select_lots = current_soup.find("select", id=re.compile("multiSelectLots"))
if select_lots:
    options = select_lots.find_all("option")
    print(f"{len(options)} option(s) trouvée(s)")
    for o in options[:20]:
        print(o.get("value"), "|", o.get_text(strip=True))
else:
    print("Aucun <select> trouvé avec cet id")

Aucun <select> trouvé avec cet id


In [100]:
patterns = [
    r'"[^"]*[Ll]ot[^"]*"\s*:\s*\[[^\]]*\]',   # un tableau JSON nommé avec "lot"
    r'Prado\.CallbackRequest\([^)]*\)',
    r'GetLots\w*\([^)]*\)',
    r'ChargerLots\w*\([^)]*\)',
]
for pat in patterns:
    found = re.findall(pat, resp_step2.text)
    print(f"'{pat}' -> {len(found)} trouvé(s)")
    for f in found[:5]:
        print("  ", f[:150])

'"[^"]*[Ll]ot[^"]*"\s*:\s*\[[^\]]*\]' -> 0 trouvé(s)
'Prado\.CallbackRequest\([^)]*\)' -> 0 trouvé(s)
'GetLots\w*\([^)]*\)' -> 0 trouvé(s)
'ChargerLots\w*\([^)]*\)' -> 0 trouvé(s)


In [101]:
import re

error_candidates = current_soup.find_all(attrs={"class": re.compile(r"error|alert|warning|invalid", re.IGNORECASE)})
print(f"{len(error_candidates)} élément(s) avec une classe d'erreur")
for e in error_candidates[:10]:
    print(e.get_text(strip=True)[:150])

1 élément(s) avec une classe d'erreur



In [102]:
# Compare le texte visible, pas la taille brute
text_before = soup_retraits.get_text(" ", strip=True)
text_after = current_soup.get_text(" ", strip=True)  # la page après le clic sur validerSelectionLots

print("Longueur texte avant:", len(text_before))
print("Longueur texte après:", len(text_after))
print("Identiques ?", text_before == text_after)

# Si différents, affiche un aperçu des deux autour du même point
if text_before != text_after:
    for i in range(min(len(text_before), len(text_after))):
        if text_before[i] != text_after[i]:
            print("Premier point de différence à l'index", i)
            print("AVANT:", text_before[max(0,i-80):i+80])
            print("APRES:", text_after[max(0,i-80):i+80])
            break

Longueur texte avant: 5911
Longueur texte après: 5911
Identiques ? True


In [103]:
keywords = ["nom", "prenom", "email", "mail", "condition", "civilite", "declar", "identit", "cgu", "accept"]

for tag in current_soup.find_all(["input", "select", "textarea", "label"]):
    name = (tag.get("name") or tag.get("for") or "").lower()
    text = tag.get_text(strip=True).lower() if tag.name == "label" else ""
    if any(k in name or k in text for k in keywords):
        print(tag.name, "|", tag.get("name") or tag.get("for"), "|", tag.get("type"), "|", (tag.get("value") or text)[:50])

In [104]:
error_el = current_soup.find(attrs={"class": re.compile(r"error|alert|warning|invalid", re.IGNORECASE)})
print("Classe:", error_el.get("class"))
print("Contenu:", error_el.get_text(strip=True))
print("\nHTML complet:")
print(error_el.prettify())

Classe: ['check-invalide']
Contenu: 

HTML complet:
<span class="check-invalide" title="Format incorrecte">
 <img alt="Format incorrecte" src="themes/images/picto-check-not-ok.gif"/>
</span>



In [105]:
error_el = current_soup.find(attrs={"class": re.compile(r"error|alert|warning|invalid", re.IGNORECASE)})
print("Classe:", error_el.get("class"))
print("Contenu:", error_el.get_text(strip=True))
print("\nHTML complet:")
print(error_el.prettify())

Classe: ['check-invalide']
Contenu: 

HTML complet:
<span class="check-invalide" title="Format incorrecte">
 <img alt="Format incorrecte" src="themes/images/picto-check-not-ok.gif"/>
</span>



In [106]:
print("Status code:", resp_step2.status_code)
print("URL finale (après redirections éventuelles):", resp_step2.url)
print("Historique de redirections:", resp_step2.history)

Status code: 200
URL finale (après redirections éventuelles): https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDetailsConsultation&refConsultation=1023160&orgAcronyme=g3h&retraits
Historique de redirections: []


In [108]:
pagestate_avant = extract_form_fields(soup_retraits).get("PRADO_PAGESTATE", "")
pagestate_envoye = form_data_lots.get("PRADO_PAGESTATE", "")
pagestate_apres = extract_form_fields(current_soup).get("PRADO_PAGESTATE", "")

print("Pagestate avant == envoyé ?", pagestate_avant == pagestate_envoye)
print("Pagestate envoyé == après ?", pagestate_envoye == pagestate_apres)
print("Longueur pagestate avant:", len(pagestate_avant))
print("Longueur pagestate après:", len(pagestate_apres))

Pagestate avant == envoyé ? True
Pagestate envoyé == après ? True
Longueur pagestate avant: 13372
Longueur pagestate après: 13372


In [109]:
ROOT_URL = "https://www.marchespublics.gov.ma"

ref_consultation = "1023160"  # à remplacer par ta vraie référence de test
org_acronyme = "g3h"

# Étape 1 : ouvrir la fiche (établit le contexte de session pour cette consultation)
detail_url = f"{ROOT_URL}/index.php?page=entreprise.EntrepriseDetailConsultation&refConsultation={ref_consultation}&orgAcronyme={org_acronyme}"
resp1 = session.get(detail_url, timeout=20)
print("1. Fiche:", resp1.status_code)

# Étape 2 : ouvrir la page de demande de téléchargement DCE
telechargement_url = f"{ROOT_URL}/index.php?page=entreprise.EntrepriseDemandeTelechargementDce&refConsultation={ref_consultation}&orgAcronyme={org_acronyme}"
resp2 = session.get(telechargement_url, headers={"Referer": detail_url}, timeout=20)
print("2. Page téléchargement:", resp2.status_code)

soup_dl = BeautifulSoup(resp2.text, "lxml")
form_data_dl = extract_form_fields(soup_dl)

# Étape 3 : soumettre le formulaire d'identité (champs identifiés dans le HAR)
form_data_dl["PRADO_POSTBACK_TARGET"] = "ctl0$CONTENU_PAGE$validateButton"
form_data_dl["PRADO_POSTBACK_PARAMETER"] = ""
form_data_dl["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$RadioGroup"] = "ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$choixTelechargement"
form_data_dl["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$accepterConditions"] = "on"
form_data_dl["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$nom"] = "a"
form_data_dl["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$prenom"] = "aze"
form_data_dl["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$email"] = "a@gmail.com"
form_data_dl["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$etablissementEntreprise"] = "ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$france"
form_data_dl["ctl0$CONTENU_PAGE$EntrepriseFormulaireDemande$pays"] = "0"

resp3 = session.post(telechargement_url, data=form_data_dl, headers={"Referer": telechargement_url}, timeout=20)
print("3. Validation identité:", resp3.status_code)

soup_after_validate = BeautifulSoup(resp3.text, "lxml")
form_data_final = extract_form_fields(soup_after_validate)

# Étape 4 : déclencher le téléchargement complet
form_data_final["PRADO_POSTBACK_TARGET"] = "ctl0$CONTENU_PAGE$EntrepriseDownloadDce$completeDownload"
form_data_final["PRADO_POSTBACK_PARAMETER"] = ""

resp4 = session.post(telechargement_url, data=form_data_final, headers={"Referer": telechargement_url}, timeout=20, allow_redirects=False)
print("4. Déclenchement téléchargement:", resp4.status_code)
print("   Location:", resp4.headers.get("Location"))

# Étape 5 : suivre la redirection pour récupérer le vrai fichier
if resp4.status_code in (301, 302) and resp4.headers.get("Location"):
    final_url = ROOT_URL + "/" + resp4.headers["Location"].lstrip("/")
    resp5 = session.get(final_url, timeout=30)
    print("5. Téléchargement final:", resp5.status_code)
    print("   Content-Type:", resp5.headers.get("Content-Type"))
    print("   Content-Disposition:", resp5.headers.get("Content-Disposition"))

    if "zip" in resp5.headers.get("Content-Type", ""):
        with open("dossier_consultation.zip", "wb") as f:
            f.write(resp5.content)
        print("   Fichier sauvegardé : dossier_consultation.zip")

1. Fiche: 200
2. Page téléchargement: 200
3. Validation identité: 200
4. Déclenchement téléchargement: 302
   Location: index.php?page=entreprise.EntrepriseDownloadCompleteDce&reference=1023160&orgAcronym=g3h
5. Téléchargement final: 200
   Content-Type: text/html; charset=utf-8
   Content-Disposition: None


In [110]:
print(final_url)

https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDownloadCompleteDce&reference=1023160&orgAcronym=g3h


In [111]:
print(resp5.url)
print(resp5.status_code)
print(resp5.text[:1000])

https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDownloadCompleteDce&reference=1023160&orgAcronym=g3h
200
<html><head><title>Request Rejected</title></head><body>The requested URL was rejected. Please consult with your administrator.<br><br>Your support ID is: 12522705939203862104<br><br><a href='javascript:history.back();'>[Go Back]</a></body></html>


In [112]:
print(session.cookies.get_dict())

{'PHPSESSID': 'f02d04538362987c1415fcc90b0ce56e', 'SERVERID': 'server15', 'dcookie': '!rC66+KSzlBHO7MobC33zLR2IQf6XfCjbTIhhimjLygN0MT5N/FW9FevQr47qvJuFLToUcuv6XQH86wk=', 'TS01fd0ca2': '01a853b00505da213892921e4aad519178ab757e11211bbe6b3b0043e25b404304134bd232dbafc7e3cf1cc0fab2dc6695bc122a075b9749470b7397cd4787e21d5f12cf3767bbb1684b30c17a7bcf949db956c1ac', 'RX01fd0ca2': '01a853b0056d53725ca23fdd10fe8ad2c1f7c6788b6562317fe110bd653fbe1d4f5b19d793e1b7646bd192b3a18eda3e455ff452a408f38962011dd132bdcdc27748847b1a'}


In [113]:
print(session.cookies.get_dict())

{'PHPSESSID': 'f02d04538362987c1415fcc90b0ce56e', 'SERVERID': 'server15', 'dcookie': '!rC66+KSzlBHO7MobC33zLR2IQf6XfCjbTIhhimjLygN0MT5N/FW9FevQr47qvJuFLToUcuv6XQH86wk=', 'TS01fd0ca2': '01a853b00505da213892921e4aad519178ab757e11211bbe6b3b0043e25b404304134bd232dbafc7e3cf1cc0fab2dc6695bc122a075b9749470b7397cd4787e21d5f12cf3767bbb1684b30c17a7bcf949db956c1ac', 'RX01fd0ca2': '01a853b0056d53725ca23fdd10fe8ad2c1f7c6788b6562317fe110bd653fbe1d4f5b19d793e1b7646bd192b3a18eda3e455ff452a408f38962011dd132bdcdc27748847b1a'}


In [114]:
resp5 = session.get(
    final_url,
    headers={
        "Referer": telechargement_url
    },
    timeout=30
)